# Doctor Prescription Reader — Simple, Free Version

**What this notebook does:** upload a photo of a handwritten prescription, get back a
full structured summary (patient info, diagnosis, medicines with dosage/frequency/
duration) — free, using Google Gemini's multimodal API.

**How calibration sheets fit in:** instead of training/fine-tuning a model on them
(expensive, needs GPU, needs lots of data), this notebook uses a few filled calibration
sheet photos as **few-shot examples** — shown to Gemini directly in the prompt, so it
sees this doctor's handwriting style before reading the real prescription. This still
satisfies "make and use a calibration sheet" — it's just a different (free, simpler,
often just as effective) way of using it.

**Run cells top to bottom.** Only Cell 2 needs anything from you (a free API key).

## 1 — Install

In [ ]:
!pip install -q -U google-genai pillow

## 2 — API key
Get a free key at **aistudio.google.com/apikey** (Google account, no card needed). Paste it below — it's only kept for this session, never saved to the notebook.

In [ ]:
from google import genai
from getpass import getpass

api_key = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=api_key)
print("Client ready.")

## 3 — Reliability helper: try several models, fall back automatically
Free-tier Gemini models occasionally return "model overloaded" errors. This tries a short list of models in order and automatically retries with backoff, so a temporary outage on one model doesn't block you.

In [ ]:
import time

MODEL_FALLBACK_LIST = ["gemini-3.6-flash", "gemini-2.5-flash-lite", "gemini-flash-latest"]

def call_gemini(contents, models=MODEL_FALLBACK_LIST, max_retries=3):
    last_error = None
    for model_name in models:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(model=model_name, contents=contents)
                print(f"(used model: {model_name})")
                return response.text
            except Exception as e:
                last_error = e
                if "UNAVAILABLE" in str(e) or "503" in str(e) or "overloaded" in str(e).lower():
                    wait = 2 ** attempt
                    print(f"{model_name} busy, retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    break  # non-overload error, try next model instead of retrying same one
    raise RuntimeError(f"All models failed. Last error: {last_error}")

## 4 — Upload calibration examples (optional but recommended)
Upload a FEW photos from your filled calibration sheets — just 2 to 5 clear examples is enough, more doesn't help much and slows things down. For each one, type the correct text exactly as written, so Gemini can see "this shape of handwriting = this word". **You can skip this step (run the empty-list cell below instead) and the notebook still works** — it'll just be reading cold, without seeing this doctor's style first.

In [ ]:
from google.colab import files
from PIL import Image

print("Upload 2-5 calibration example images (single words or short lines):")
calibration_uploads = files.upload()
calibration_files = list(calibration_uploads.keys())
print(f"Uploaded: {calibration_files}")

In [ ]:
# For each uploaded calibration image, type the CORRECT text it shows.
# Edit this dictionary to match your uploads (filename -> correct text).
calibration_labels = {
    # "example1.jpg": "Dolo 650",
    # "example2.jpg": "Tab. Hamlo 5mg BD 30 days",
}

calibration_examples = []
for fname in calibration_files:
    correct_text = calibration_labels.get(fname, None)
    if correct_text is None:
        print(f"WARNING: no label given for {fname} in calibration_labels above — skipping it.")
        continue
    calibration_examples.append((Image.open(fname), correct_text))

print(f"Using {len(calibration_examples)} labeled calibration examples.")

**If you skipped calibration uploads**, run this cell instead to continue with zero examples:

In [ ]:
# Uncomment and run this line ONLY if you skipped the upload step above:
# calibration_examples = []

## 5 — Upload the real prescription to decode

In [ ]:
print("Upload the prescription photo you want decoded:")
uploaded = files.upload()
prescription_path = list(uploaded.keys())[0]
prescription_image = Image.open(prescription_path)
print("Using:", prescription_path)
prescription_image

## 6 — Build the prompt (calibration examples + the real prescription)
This asks for one thing: a full, structured, human-readable summary — patient info, diagnosis, vitals, and every medicine with dosage/frequency/duration — plus a confidence flag on anything it's not sure about, so nothing wrong gets silently trusted.

In [ ]:
instruction_text = """You are reading a real, handwritten doctor's prescription (likely
Indian medical practice — expect Indian brand-name medicines like Dolo 650, Augmentin,
Azithral, Glycomet, etc., and shorthand like "1-0-1" for dosage timing, "OD/BD/TDS" for
frequency).

If example images of this SAME doctor's handwriting are provided first (with their
correct readings), use them to calibrate your understanding of this doctor's letter
shapes before reading the actual prescription that follows.

Read the actual prescription and return ONLY this JSON structure, nothing else:

{
  "patient_info": {"name": "...", "age_gender": "...", "date": "..."},
  "diagnosis": ["...", "..."],
  "vitals_investigations": ["...", "..."],
  "medicines": [
    {"medicine_name": "...", "dosage": "...", "frequency": "...", "duration": "...",
     "confidence": "high | medium | low",
     "needs_review": true or false}
  ],
  "other_notes": "..."
}

Mark needs_review true for ANYTHING you are not confident about, rather than guessing
silently. Use null for fields you genuinely cannot determine. Do not include any text
outside the JSON object."""

contents = [instruction_text]

for img, label in calibration_examples:
    contents.append(f"Example — this doctor wrote:")
    contents.append(img)
    contents.append(f"Correct reading: {label}")

contents.append("Now read this actual prescription:")
contents.append(prescription_image)

print(f"Prompt built with {len(calibration_examples)} calibration examples + 1 real prescription.")

## 7 — Call Gemini (with automatic fallback/retry)

In [ ]:
response_text = call_gemini(contents)
print(response_text)

## 8 — Parse and print a clean, readable summary

In [ ]:
clean_text = response_text.strip()
if clean_text.startswith("```"):
    clean_text = clean_text.split("```")[1]
    if clean_text.startswith("json"):
        clean_text = clean_text[4:]

prescription_summary = json.loads(clean_text.strip())

# --- Human-readable printout ---
p = prescription_summary.get("patient_info", {})
print("=" * 55)
print("PRESCRIPTION SUMMARY")
print("=" * 55)
print(f"Patient   : {p.get('name')}  |  {p.get('age_gender')}  |  {p.get('date')}")

print("\nDiagnosis:")
for d in prescription_summary.get("diagnosis", []) or []:
    print(f"  - {d}")

print("\nVitals / Investigations:")
for v in prescription_summary.get("vitals_investigations", []) or []:
    print(f"  - {v}")

print("\nMedicines:")
for i, med in enumerate(prescription_summary.get("medicines", []) or [], 1):
    flag = "  <-- PLEASE VERIFY AGAINST ORIGINAL IMAGE" if med.get("needs_review") else ""
    print(f"  {i}. {med.get('medicine_name')} — {med.get('dosage')} — "
          f"{med.get('frequency')} — {med.get('duration')} "
          f"[confidence: {med.get('confidence')}]{flag}")

notes = prescription_summary.get("other_notes")
if notes:
    print(f"\nOther notes: {notes}")

print("=" * 55)
print("\nFull JSON:")
print(json.dumps(prescription_summary, indent=2))

## 9 — (Optional) Save the summary to a file

In [ ]:
with open("prescription_summary.json", "w") as f:
    json.dump(prescription_summary, f, indent=2)

from google.colab import files
files.download("prescription_summary.json")

---
## Notes on accuracy and what to do next

- This will not be perfect on every prescription — treat any `needs_review: true` field
  as "a human must check the original image before trusting this," always, especially
  for medicine names and dosages.
- Adding real calibration examples in Cell 4 (2-5 clear ones from THIS doctor) should
  help accuracy on THIS doctor's handwriting specifically — worth testing with and
  without them on the same prescription to see the actual difference.
- If accuracy on real, messy handwriting still isn't where you need it, the next lever
  is more/better calibration examples (not more code) — this approach scales with
  example quality, not engineering complexity.